### HWP로더 : page 284

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# !uv add olefile langchain-core langchain-community

In [3]:
import re
import struct
import zlib
from collections.abc import Iterator
from pathlib import Path

import olefile
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

In [ ]:

class HWPLoader(BaseLoader):
    """HWP 5.x 파일에서 본문 텍스트를 추출합니다."""

    FILE_HEADER = "FileHeader"
    HWP_SUMMARY = "\u0005HwpSummaryInformation"

    def __init__(self, file_path: str) -> None:
        self.file_path = Path(file_path)

    def _is_valid_hwp(self, ole: olefile.OleFileIO) -> bool:
        streams = {"/".join(item) for item in ole.listdir()}
        return self.FILE_HEADER in streams and self.HWP_SUMMARY in streams

    def _is_compressed(self, ole: olefile.OleFileIO) -> bool:
        header = ole.openstream(self.FILE_HEADER).read()
        return bool(header[36] & 1)

    @staticmethod
    def _extract_text(section: bytes) -> str:
        texts: list[str] = []
        position = 0

        while position + 4 <= len(section):
            header = struct.unpack_from("<I", section, position)[0]
            record_type = header & 0x3FF
            record_length = (header >> 20) & 0xFFF
            position += 4

            if record_length == 0xFFF:
                if position + 4 > len(section):
                    break
                record_length = struct.unpack_from("<I", section, position)[0]
                position += 4

            payload = section[position : position + record_length]
            position += record_length

            # HWPTAG_PARA_TEXT의 태그 ID는 67입니다.
            if record_type == 67:
                text = payload.decode("utf-16-le", errors="ignore")
                text = re.sub(r"[\x00-\x08\x0b-\x1f]", "", text)
                texts.append(text)

        return "\n".join(texts)

    def lazy_load(self) -> Iterator[Document]:
        with olefile.OleFileIO(self.file_path) as ole:
            if not self._is_valid_hwp(ole):
                raise ValueError(f"유효한 HWP 파일이 아닙니다: {self.file_path}")

            compressed = self._is_compressed(ole)
            sections = sorted(
                (item for item in ole.listdir() if item[0] == "BodyText"),
                key=lambda item: int(item[1][len("Section") :]),
            )

            for section_number, section_path in enumerate(sections):
                data = ole.openstream(section_path).read()
                if compressed:
                    data = zlib.decompress(data, -15)

                yield Document(
                    page_content=self._extract_text(data),
                    metadata={
                        "source": str(self.file_path),
                        "section": section_number,
                    },
                )

In [6]:
loader = HWPLoader("../data/디지털 정부혁신 추진계획.hwp")
documents = loader.load()

In [9]:

print(documents[0].page_content[:200])
print("------------")
print(f"섹션 수: {len(documents)}")
print("------------")
print(documents[0].page_content[:1000])

捤獥汤捯
漠杳
디지털 정부혁신 추진계획
2019. 10. 29.
      氠瑢桤灧
漠杳
관계부처 합동
桤灧桤灧
氠瑢桤灧漠杳
순    서
Ⅰ. 개요	翌ȃ	 1
Ⅱ. 디지털 정부혁신 추진계획	ㆬȃ	 2
  1. 우선 추진과제	枈ȃ	 2
     ① 선제적·통합적 대국민 서비스 혁신
     ② 공공부문 마이데이터 활성화
     ③ 시민참여를 위한 플랫폼 고
------------
섹션 수: 1
------------
捤獥汤捯
漠杳
디지털 정부혁신 추진계획
2019. 10. 29.
      氠瑢桤灧
漠杳
관계부처 합동
桤灧桤灧
氠瑢桤灧漠杳
순    서
Ⅰ. 개요	翌ȃ	 1
Ⅱ. 디지털 정부혁신 추진계획	ㆬȃ	 2
  1. 우선 추진과제	枈ȃ	 2
     ① 선제적·통합적 대국민 서비스 혁신
     ② 공공부문 마이데이터 활성화
     ③ 시민참여를 위한 플랫폼 고도화
     ④ 현장중심 협업을 지원하는 스마트 업무환경 구현
     ⑤ 클라우드와 디지털서비스 이용 활성화
     ⑥ 개방형 데이터·서비스 생태계 구축
  2. 중장기 범정부 디지털 전환 로드맵 수립	ᲈȃ	 4
Ⅲ. 추진체계 및 일정	僬ȃ	 4
 [붙임] 디지털 정부혁신 우선 추진과제(상세)	ᬜȃ	 8
氠瑢漠杳
Ⅰ. 개 요
□ 추진 배경湯湷
 ○ 우리나라는 국가적 초고속 정보통신망 투자와 적극적인 공공정보화 사업 추진에 힘입어 세계 최고수준의 전자정부를 구축‧운영
     * UN전자정부평가에서 2010‧12‧14년 1위, 16‧18년 3위, UN공공행정상 13회 수상
 ○ 그러나, 인공지능‧클라우드 중심의 디지털 전환(Digital Transformation) 
시대가 도래함에 따라 기존 전자정부의 한계 표출
   - 축적된 행정데이터에도 불구하고 기관간 연계‧활용 미흡, 부처 단위로 단절된 서비스, 신기술 활용을 위한 제도‧기반 부족
   - 디지털 전환을 위한 컨트롤타워가 없고, 구체적 전략도 부재
 ○ 이에, ‘19.3월부터 공공부문 ICT 활용현황 및 문제점 검토에 착수하여 공공분